# AI Research Paper Intelligent System (with Memory-Enabled Agent)


**Problem** - help a researcher find relevant papers quickly

**Solution** - semantic search over paper embeddings, summarization, keyword extraction, and an LLM agent that can use these as tools *and* remember things about the user across turns.

**Pipeline**
1. Data collection
2. Text extraction / preprocessing
3. Embedding generation (Sentence-Transformers)
4. Vector store (FAISS)
5. Semantic search
6. Summarization + keyword extraction
7. Agentic AI (LangChain agent with tools)
8. **Long-term memory** for the agent (save / recall facts about the user)


## 0. Setup

In [1]:
!pip install -q datasets sentence-transformers faiss-cpu "transformers==4.46.3" keybert langchain-core langchain-community langchain-huggingface langchain_groq langchain tenacity groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [2]:
import os
import pickle
import uuid
import numpy as np
import pandas as pd
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from keybert import KeyBERT

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from groq import RateLimitError, BadRequestError

from google.colab import userdata


## 1. Data collection

In [3]:

dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


## 2. Text extraction & preprocessing

In [4]:
df = pd.DataFrame(dataset)[["title", "abstract"]].head(15000).copy()
df["paper_text"] = (
    (df["title"] + " " + df["abstract"])
    .fillna("")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

print(df.shape)
df.head()


(15000, 3)


,title,abstract,paper_text
0,Learning from compressed observations,The problem of statistical learning is to co...,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...,Parametric Learning and Monte Carlo Optimizati...


## 3. Sentence embeddings

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

EMBEDDING_PATH = "embedding.npy"

if os.path.exists(EMBEDDING_PATH):
    print("Loading saved embeddings")
    embeddings = np.load(EMBEDDING_PATH)
else:
    print("Generating embeddings")
    embeddings = embedding_model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save(EMBEDDING_PATH, embeddings)
    print("Embeddings saved successfully!")

print(embeddings.shape, embeddings.dtype)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading saved embeddings
(15000, 384) float32


## 4. Vector store (FAISS)

> Note: this index is renamed `paper_index` (instead of the generic `index`) so it can never collide with the separate memory index built later.

In [6]:
PAPER_INDEX_PATH = "paper_faiss.index"

if os.path.exists(PAPER_INDEX_PATH):
    print("Loading existing FAISS index")
    paper_index = faiss.read_index(PAPER_INDEX_PATH)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    paper_index = faiss.IndexFlatIP(embeddings.shape[1])
    paper_index.add(embeddings)
    faiss.write_index(paper_index, PAPER_INDEX_PATH)
    print("FAISS index saved successfully!")


Loading existing FAISS index


## 5. Semantic search

In [7]:
def search_paper(query, k=5):
    """Return (scores, indices) of the top-k most similar papers to `query`."""
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = paper_index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print(f"Similarity: {round(float(score), 4)}")
        print(f"Title: {df.iloc[idx]['title']}")
        print()
    return D, I

_ = search_paper("deep learning for medical image analysis")


Similarity: 0.6807
Title: A Perspective on Deep Imaging

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Similarity: 0.6522
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Similarity: 0.6281
Title: High-Resolution Breast Cancer Screening with Multi-View Deep
  Convolutional Neural Networks

Similarity: 0.6131
Title: Deep Learning for the Classification of Lung Nodules



## 6. Summarization

In [8]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_abstract(abstract, max_length=120, min_length=40):
    return summarizer(
        abstract, max_length=max_length, min_length=min_length, do_sample=False
    )[0]["summary_text"]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


## 7. Keyword extraction

In [9]:
kw_model = KeyBERT(embedding_model)

def extract_keywords(text, top_n=5):
    return kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_n
    )


## 8. Combined search + summarize + keywords

In [10]:
def search_and_summarize(query, k=5):
    """Search papers, then summarize and extract keywords for each hit."""
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = paper_index.search(query_embedding, k)

    results = []
    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
        paper = df.iloc[idx]
        summary = summarize_abstract(paper["abstract"])
        keywords = extract_keywords(paper["abstract"], top_n=5)

        results.append({
            "rank": rank,
            "similarity": round(float(score), 4),
            "title": paper["title"],
            "abstract": paper["abstract"],
            "summary": summary,
            "keywords": keywords
        })
    return results

for r in search_and_summarize("Deep Learning in Medical Imaging", k=3):
    print("=" * 100)
    print(f"Rank {r['rank']} | Similarity {r['similarity']}")
    print("Title:", r["title"])
    print("Summary:", r["summary"])
    print("Keywords:", r["keywords"])
    print()


Rank 1 | Similarity 0.7355
Title: A Perspective on Deep Imaging
Summary: The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.
Keywords: [('imaging deep', 0.5919), ('medical imaging', 0.5281), ('tomographic imaging', 0.525), ('image reconstruction', 0.5095), ('imaging', 0.4528)]

Rank 2 | Similarity 0.6882
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection
Summary: The classification of MRI images according to the anatomical field of view is a necessary task to solve when faced with the increasing quantity of medical images. Using a common architecture (such as AlexNet) provides quite good results, but not sufficient for clinical use.
Keywords: [('classific

## 9. Agentic AI — LLM setup

In [11]:
api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0
)

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(RateLimitError)
)
def invoke_llm_with_backoff(prompt):
    """Invoke the LLM with exponential backoff on rate-limit errors."""
    return llm.invoke(prompt)

response = invoke_llm_with_backoff("Hello, who are you?")
print(response.content)


I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and respond to human language in a way that's natural and helpful.

I don't have a personal identity or emotions like humans do, but I'm here to assist you with any questions, topics, or tasks you'd like to discuss. I can provide information, answer questions, generate text, and even engage in conversation.

I'm constantly learning and improving my responses based on the data and interactions I receive, so feel free to ask me anything and see how I can help!


## 10. Long-term memory

A small persistent memory store for the agent: facts are saved to disk (`memory_store.pkl`) and retrieved by semantic similarity through their own FAISS index (`memory_index`), kept completely separate from `paper_index` above.

In [12]:
MEMORY_PATH = "memory_store.pkl"

memory_documents = []
if os.path.exists(MEMORY_PATH):
    with open(MEMORY_PATH, "rb") as f:
        memory_documents = pickle.load(f)


def _persist_memory():
    with open(MEMORY_PATH, "wb") as f:
        pickle.dump(memory_documents, f)


def save_memory(text, metadata=None):
    """Store a new fact / preference in long-term memory."""
    memory_documents.append({
        "id": str(uuid.uuid4()),
        "text": text,
        "metadata": metadata or {}
    })
    _persist_memory()
    return f"Saved to memory: {text}"


def retrieve_memory(query, k=3):
    """Return the k most relevant stored memories for a query."""
    if len(memory_documents) == 0:
        return []

    query_embedding = embedding_model.encode([query]).astype(np.float32)
    memory_embeddings = embedding_model.encode(
        [m["text"] for m in memory_documents]
    ).astype(np.float32)

    faiss.normalize_L2(query_embedding)
    faiss.normalize_L2(memory_embeddings)

    memory_index = faiss.IndexFlatIP(memory_embeddings.shape[1])
    memory_index.add(memory_embeddings)

    k = min(k, len(memory_documents))
    D, I = memory_index.search(query_embedding, k)

    return [memory_documents[idx] for idx in I[0]]


def build_memory_context(query, k=3):
    """Format retrieved memories as plain text context for the LLM."""
    memories = retrieve_memory(query, k=k)
    if not memories:
        return ""
    return "\n".join(f"- {m['text']}" for m in memories)


### Expose memory to the agent as tools

In [13]:
@tool
def remember_fact(text: str) -> str:
    """Save an important fact, preference, or piece of context about the user to long-term memory, so it can be recalled in future conversations."""
    return save_memory(text)


@tool
def recall_memory(query: str) -> str:
    """Search long-term memory for facts relevant to the query and return them."""
    context = build_memory_context(query)
    return context if context else "No relevant memories found."


### Research tools

In [14]:
@tool
def search_and_summarize_papers(query: str, k: int = 5) -> str:
    """Search research papers from the FAISS database, retrieve the top-k most similar papers, summarize each with BART, and return formatted results."""
    results = search_and_summarize(query, k=k)
    output = ""
    for r in results:
        output += f"Rank: {r['rank']}\n"
        output += f"Similarity Score: {r['similarity']}\n"
        output += f"Title: {r['title']}\n\n"
        output += r["summary"] + "\n\n"
    return output


@tool
def extract_keywords_tool(text: str, top_n: int = 5) -> str:
    """Extract the most important keywords from the given text using KeyBERT."""
    keywords = extract_keywords(text, top_n=top_n)
    result = "Top Keywords:\n\n"
    for rank, (keyword, score) in enumerate(keywords, start=1):
        result += f"{rank}. {keyword} (Relevance Score: {round(score, 4)})\n"
    return result


@tool
def compare_papers(paper1: str, paper2: str) -> str:
    """Compare two research papers based on their abstracts, retrieved via semantic search, using the LLM to produce a structured comparison table."""
    embedding1 = embedding_model.encode([paper1])
    faiss.normalize_L2(embedding1)
    _, I1 = paper_index.search(embedding1, 1)
    first_paper = df.iloc[I1[0][0]]

    embedding2 = embedding_model.encode([paper2])
    faiss.normalize_L2(embedding2)
    _, I2 = paper_index.search(embedding2, 1)
    second_paper = df.iloc[I2[0][0]]

    comparison_prompt = f"""
Compare the following two research papers.

Paper 1
Title: {first_paper['title']}
Abstract: {first_paper['abstract']}

Paper 2
Title: {second_paper['title']}
Abstract: {second_paper['abstract']}

Compare them based on:
1. Research Objective
2. Methodology
3. Key Contributions
4. Advantages
5. Limitations
6. Applications

Present the comparison in a clear table.
"""

    return llm.invoke(comparison_prompt).content


## 11. Build the memory-enabled agent

In [15]:
tools = [
    search_and_summarize_papers,
    extract_keywords_tool,
    compare_papers,
    remember_fact,
    recall_memory
]

agent = create_agent(model=llm, tools=tools)


In [16]:
@retry(
    wait=wait_exponential(multiplier=1, min=2, max=15),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type((RateLimitError, BadRequestError))
)
def _invoke_agent_with_backoff(messages):
    return agent.invoke({"messages": messages})

## 12. Memory-aware invocation wrapper

Every call to `ask_agent` retrieves relevant long-term memories and injects them as context, *and* the agent itself can call `remember_fact` / `recall_memory` whenever it decides to — so memory works both automatically (context injection) and on the agent's own initiative (tool use).

In [17]:
def ask_agent(user_query, k_memories=3):
    """Ask the agent a question with relevant long-term memory injected as context."""
    memory_context = build_memory_context(user_query, k=k_memories)

    enhanced_query = f"""Relevant Long-Term Memory:
{memory_context if memory_context else "None"}

Current User Question:
{user_query}
"""

    response = _invoke_agent_with_backoff(
        [{"role": "user", "content": enhanced_query}]
    )

    return response["messages"][-1].content


## 13. Testing

In [18]:
print(save_memory("User likes transformer papers.", {"type": "preference"}))
print(save_memory("User is working on BERT.", {"type": "research"}))


Saved to memory: User likes transformer papers.
Saved to memory: User is working on BERT.


In [19]:
memory_context = build_memory_context("Recommend recent Transformer papers")
enhanced_query = f"""Relevant Long-Term Memory:
{memory_context if memory_context else "None"}

Current User Question:
Recommend recent Transformer papers
"""

response = _invoke_agent_with_backoff([{"role": "user", "content": enhanced_query}])

for m in response["messages"]:
    print(type(m).__name__, "-" * 40)
    print(m.content)
    print()

Your max_length is set to 120, but your input_length is only 11. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)


HumanMessage ----------------------------------------
Relevant Long-Term Memory:
- User likes transformer papers.
- User is working on BERT.

Current User Question:
Recommend recent Transformer papers


AIMessage ----------------------------------------


ToolMessage ----------------------------------------
- User likes transformer papers.
- User is working on BERT.

ToolMessage ----------------------------------------
Rank: 1
Similarity Score: 0.2933
Title: cvpaper.challenge in 2015 - A review of CVPR2015 and DeepSurvey

The "cvpaper.challenge" is a group composed of members from AIST, Tokyo DenkiUniv. (TDU), and Univ. of Tsukuba. It aims to systematically summarize papers on computer vision, pattern recognition, and related fields. For this particular review, we focused on reading the ALL 602 conference papers at the CVPR2015.

Rank: 2
Similarity Score: 0.2836
Title: Dense Transformer Networks

The key idea of current deep learning methods for dense prediction is toapply a model on a

In [ ]:
print(ask_agent("Remember that I'm especially interested in vision transformers for medical imaging."))


In [ ]:
print(ask_agent("What do you know about my research interests?"))
